# Forecasting 4 — CRISP-DM Phases 1–6: Business-to-Deployment Workshop

**Business context:** Garden-centre network

**Problem:** Forecast next-week store sales to plan labour and seasonal stock.

**Proposed success criterion:** Beat the previous-week naive forecast MAE by at least 10% on the last 12 weeks; examine seasonal peaks.

**Dataset:** `forecasting_4.csv`. These are synthetic teaching records, not real company data.

**Submission deadline: Monday 5 October 2026 at 13:00 (1:00 PM).** Submit your completed team notebook on Brightspace. Phases 1–2 must be reviewed and updated; your main practical work is Phases 3–6. Any unfinished in-class work must be completed before the deadline.


### Phase 1 – Business understanding
- **Decision:** how many staff hours to roster and how much seasonal stock to order per store for next week.
- **Stakeholder:** store managers (rosters) and the central planning/purchasing team (stock).
- **Prediction moment:** end of week *t−1* (Sunday evening), forecasting total sales for week *t*.
- **Observation unit / key:** one store-week, key = (`store_id`, `week_start`). **Target:** `sales_eur`.
- **Revised success criterion:** the supplied criterion (beat previous-week naive MAE by ≥10% on the last 12 weeks) is kept, but it hides peaks, which is where labour/stock errors cost most. We add: (a) the model must also beat naive on holiday/promo weeks, and (b) no store may be worse than naive, with bias checked (under-forecasting → empty shelves/understaffing).

### Phase 2 – Data understanding
**Findings (evidence below):**
1. Strong seasonality: mean sales peak in Dec–Feb (~€11.9–12.4k) and dip in Jul–Aug (~€9.1k); holiday weeks average ~€12.8k vs ~€10.4k. Calendar features are worth adding.
2. `footfall_actual` (r = 0.72) and `returns_eur_after_week` (r = 0.52) correlate strongly with sales, but both are only known *after* the week → leakage, must be excluded.

**Limitations:**
1. Data-quality issues: one duplicated store-week (S02, 2024-05-20), a negative sales value (S03, −€220), and inconsistent labels (`PROMO`/`promo`, `South`/`south`).
2. Only 6 stores and ~2 years (105 weeks), synthetic data, and no cost information (€ per staff hour, cost of a stock-out) to translate MAE into business value.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA = Path('forecasting_4.csv')
if not DATA.exists(): DATA = Path('forecasting_4.csv')
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))
print('Exact duplicates:',df.duplicated().sum())

(631, 12)


,store_id,week_start,promo_label,holiday_week,footfall_actual,rain_mm_actual,store_floor_m2,store_size_category,average_price_eur,returns_eur_after_week,sales_eur,store_zone
0,S01,2024-01-01,PROMO,0,196,6.0,332,Small,15.39,173.30,8143.14,North
1,S01,2024-01-08,regular,0,223,10.3,332,Small,15.11,214.60,6664.28,North
2,S01,2024-01-15,regular,0,262,12.2,332,Small,14.52,305.87,6970.34,North
3,S01,2024-01-22,regular,0,177,0.0,332,Small,15.01,318.84,7302.29,North
4,S01,2024-01-29,regular,0,227,8.9,332,Small,15.97,261.98,7350.22,North


store_id                  0
week_start                0
promo_label               0
holiday_week              0
footfall_actual           0
rain_mm_actual            0
store_floor_m2            0
store_size_category       0
average_price_eur         0
returns_eur_after_week    0
sales_eur                 0
store_zone                0
dtype: int64

Exact duplicates: 1


In [4]:
# Extra Phase 2 evidence used in the review above
print('Duplicate (store_id, week_start) keys:', df.duplicated(['store_id','week_start']).sum())
print('Category spellings:', df['promo_label'].unique(), df['store_zone'].unique())
print('Sales <= 0:\n', df.loc[df['sales_eur'] <= 0, ['store_id','week_start','sales_eur']])
print(df[['footfall_actual','returns_eur_after_week','holiday_week','sales_eur']].corr()['sales_eur'].round(2))
print(df.groupby(pd.to_datetime(df['week_start']).dt.month)['sales_eur'].mean().round(0))


Duplicate (store_id, week_start) keys: 1
Category spellings: <ArrowStringArray>
['PROMO', 'regular', 'promo', 'REGULAR']
Length: 4, dtype: str <ArrowStringArray>
['North', 'South', 'south']
Length: 3, dtype: str
Sales <= 0:
     store_id  week_start  sales_eur
272      S03  2025-03-03     -220.0
footfall_actual           0.72
returns_eur_after_week    0.52
holiday_week              0.21
sales_eur                 1.00
Name: sales_eur, dtype: float64
week_start
1     11861.0
2     11872.0
3     11545.0
4     10741.0
5     10054.0
6      9272.0
7      9120.0
8      9071.0
9      9463.0
10    10191.0
11    10914.0
12    12357.0
Name: sales_eur, dtype: float64


## Phase 3 — Data Preparation: three justified decisions
For EACH insight: evidence → technique → before/after verification → interpretation → next decision. Consider identifiers, leakage, duplicate keys, suspicious values, categories, date parsing and feature engineering. Explain why you retain, delete or combine variables. **Do not use the target or post-outcome fields as predictors.**

In [7]:
work = df.copy()

# Decision 1 – keys & dates: parse dates, drop the exact duplicate row, check key is unique
work['week_start'] = pd.to_datetime(work['week_start'])
print('Rows before:', len(work), '| duplicate keys:', work.duplicated(['store_id','week_start']).sum())
work = work.drop_duplicates().sort_values(['store_id','week_start']).reset_index(drop=True)
print('Rows after: ', len(work), '| duplicate keys:', work.duplicated(['store_id','week_start']).sum())

# Decision 2 – inconsistent categories: harmonise case
work['promo_label'] = work['promo_label'].str.lower()
work['store_zone'] = work['store_zone'].str.title()
print(work['promo_label'].value_counts().to_dict(), work['store_zone'].value_counts().to_dict())

# Decision 3 – impossible target value: negative sales -> missing, interpolate within store
bad = work['sales_eur'] < 0
print('Negative sales rows:', bad.sum())
work['sales_flag_fixed'] = bad.astype(int)
work.loc[bad, 'sales_eur'] = np.nan
work['sales_eur'] = work.groupby('store_id')['sales_eur'].transform(lambda s: s.interpolate())
print('Negative/missing after:', (work['sales_eur'] < 0).sum(), work['sales_eur'].isna().sum())

# Feature engineering: only information available at the prediction moment
# (end of week t-1 -> forecast week t). Lags use past sales only.
g = work.groupby('store_id')['sales_eur']
work['lag1']  = g.shift(1)
work['lag2']  = g.shift(2)
work['roll4'] = g.transform(lambda s: s.shift(1).rolling(4).mean())
woy = work['week_start'].dt.isocalendar().week.astype(int)
work['woy_sin'] = np.sin(2*np.pi*woy/52); work['woy_cos'] = np.cos(2*np.pi*woy/52)
work['promo'] = (work['promo_label'] == 'promo').astype(int)

# Excluded: footfall_actual & rain_mm_actual (only known after the week), returns_eur_after_week
# (post-outcome leakage), average_price_eur (realised during the week), store_size_category
# (redundant with store_floor_m2), sales_flag_fixed (audit only).
display(work[['store_id','week_start','sales_eur','lag1','roll4']].head(3))


Rows before: 631 | duplicate keys: 1
Rows after:  630 | duplicate keys: 0
{'regular': 507, 'promo': 123} {'North': 315, 'South': 315}
Negative sales rows: 1
Negative/missing after: 0 0


,store_id,week_start,sales_eur,lag1,roll4
0,S01,2024-01-01,8143.14,NaN,NaN
1,S01,2024-01-08,6664.28,8143.14,NaN
2,S01,2024-01-15,6970.34,6664.28,NaN


### Preparation decision log
| Phase 2 finding | Technique and reason | Verification | Risk | Next decision |
|---|---|---|---|---|
| 1 duplicated row (S02, 2024-05-20); dates stored as text | Parse `week_start` as date, `drop_duplicates()`, sort by store/week – lag features need exactly one row per store-week | Rows 631 → 630; duplicate keys 1 → 0 | Low – rows were exact copies | Key is unique, safe to build lags |
| Inconsistent labels (`PROMO`/`promo`, `South`/`south`) | Lower/title-case harmonisation – otherwise one-hot encoding splits one category in two | promo_label 4 → 2 levels; store_zone 3 → 2 | Low | Use binary `promo` feature |
| Sales of −€220 (S03, 2025-03-03) – impossible for a week of store sales | Set to missing and interpolate within the store (keeps the series continuous for lags); flag kept for audit | 0 negative / 0 missing after | Medium – true value unknown; could be a refund booking | Ask finance what the value means |
| Footfall, rain and returns are *actuals*; price is realised in-week | **Excluded** as predictors (leakage at the prediction moment). `store_size_category` dropped as redundant with `store_floor_m2` | Not in feature list | Losing useful signal | Add *forecasted* footfall/weather later if available |
| Seasonality + dependence on recent level | Engineered `lag1`, `lag2`, `roll4` (past sales only, shifted), week-of-year sin/cos, `holiday_week`, `promo` | First 4 weeks per store dropped (no history) | Promo assumed planned in advance | Validate promo-planning timing with marketing |

## Phase 4 — Modelling
- **Prediction moment:** Sunday after week *t−1*. **Allowed predictors:** past sales (`lag1`, `lag2`, 4-week rolling mean), calendar (week-of-year sin/cos, `holiday_week`), planned `promo`, store (`store_id`, `store_floor_m2`).
- **Excluded:** `footfall_actual`, `rain_mm_actual`, `returns_eur_after_week`, `average_price_eur` (not known at the prediction moment / post-outcome), `sales_eur` itself.
- **Split:** chronological – last 12 weeks (13 Oct – 29 Dec 2025) as test, everything before as training. Scaler and encoder are fitted inside a `Pipeline` on training data only.
- **Baseline:** naive previous-week forecast. **Candidate:** Ridge regression (simple, stable, interpretable with ~530 training rows).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

num = ['lag1','lag2','roll4','woy_sin','woy_cos','holiday_week','promo','store_floor_m2']
cat = ['store_id']
model_df = work.dropna(subset=['lag1','lag2','roll4']).copy()   # first 4 weeks per store lack history

# Chronological split: last 12 weeks = test
cutoff = model_df['week_start'].sort_values().unique()[-12]
train = model_df[model_df['week_start'] < cutoff]
test  = model_df[model_df['week_start'] >= cutoff].copy()
print('Train:', train['week_start'].min().date(), '→', train['week_start'].max().date(), len(train))
print('Test: ', test['week_start'].min().date(), '→', test['week_start'].max().date(), len(test))

pipe = Pipeline([
    ('prep', ColumnTransformer([('num', StandardScaler(), num),
                                ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])),
    ('ridge', Ridge(alpha=1.0))])
pipe.fit(train[num+cat], train['sales_eur'])   # scaler/encoder learned on training data only

test['pred_naive'] = test['lag1']             # baseline: previous week
test['pred_model'] = pipe.predict(test[num+cat])


In [ ]:
mae_naive = mean_absolute_error(test['sales_eur'], test['pred_naive'])
mae_model = mean_absolute_error(test['sales_eur'], test['pred_model'])
print(f'Naive MAE: €{mae_naive:,.0f} | Ridge MAE: €{mae_model:,.0f} | improvement: {1-mae_model/mae_naive:.1%}')

test['peak'] = np.where(test['holiday_week'].eq(1) | test['promo'].eq(1), 'holiday/promo', 'normal')
seg = lambda col: test.groupby(col).apply(lambda d: pd.Series({
        'n': len(d),
        'MAE_naive': mean_absolute_error(d['sales_eur'], d['pred_naive']),
        'MAE_model': mean_absolute_error(d['sales_eur'], d['pred_model']),
        'bias_model': (d['pred_model'] - d['sales_eur']).mean()}), include_groups=False).round(0)
display(seg('peak')); display(seg('store_id'))


## Phase 5 — Evaluation
| Test set (72 store-weeks) | Naive MAE | Ridge MAE |
|---|---|---|
| Overall | €1,058 | **€705 (−33%)** |
| Holiday/promo weeks | €1,185 | €785 |
| Normal weeks | €968 | €648 |

- The model beats the naive baseline by **33%**, well past the 10% target, and also on the peak (holiday/promo) weeks – so criteria (a) is met.
- **Criterion (b) fails for store S01**: MAE €866 vs naive €771, with a +€531 bias (systematic over-forecasting → overstaffing/overstock at the smallest store). The pooled model probably borrows too much of the larger stores' seasonal swing.
- Is the criterion realistic? 10% is easy to beat here because the naive forecast ignores seasonality and holidays; a stricter benchmark would be a seasonal-naive (same week last year). The 12-week test covers only one Christmas peak, so a rolling-origin backtest over more weeks is advisable.
- **Assumption:** no cost data is available; we assume over- and under-forecasting cost the same. Planning should supply € per staff hour and estimated stock-out cost so MAE can be converted into € impact.

## Phase 6 — Deployment proposal (design only)
Every Sunday night a scheduled job loads the closed week's sales, rebuilds the lag features and produces a next-week forecast per store. Store managers see it Monday morning in the rostering sheet and use it to set staff hours; central purchasing uses the sum across stores for seasonal stock orders. Managers can **override** the forecast (e.g. local event, road works) and must give a reason; overrides are logged and compared with actuals. **Monitoring:** weekly rolling 4-week MAE and bias per store versus the naive forecast. **Refresh trigger:** retrain monthly, or immediately if the 4-week MAE is more than 20% above the test MAE. **Pause condition:** if the model is worse than naive for a store for 4 consecutive weeks (S01 is currently borderline), that store falls back to the naive forecast until reviewed.

## Final group decision + individual accountability
**Group decision: Pilot** (stores S02–S06), *Improve* for S01. Evidence: Ridge cuts MAE by 33% overall (€1,058 → €705) and also wins on holiday/promo peak weeks, without using any leaking fields. S01 is still worse than naive with a +€531 bias, so it stays on the naive forecast while we test a store-specific scale/interaction term. Before full rollout we need cost data and a longer rolling backtest.

**[Name 1]:** *Preparation choice* – … *Model finding* – … *Business implication* – …

**[Name 2]:** *Preparation choice* – … *Model finding* – … *Business implication* – …

**[Name 3]:** *Preparation choice* – … *Model finding* – … *Business implication* – …